# 1. Project Overview

MermaidGenerate is a local-first and Google Colab-compatible AI web app for generating Mermaid **Mind Map** and **Venn Diagram** code. The primary runtime is Hugging Face **Transformers + PyTorch**, not paid APIs. Fine-tuning is supported through **LoRA**, **QLoRA 4-bit**, and **Full Fine-Tuning** paths using PEFT.

Default base model: `TinyLlama/TinyLlama-1.1B-Chat-v1.0`.

The Gradio app provides two tabs: **Generator Mermaid** and **Dataset & Fine-Tuning**.

## Local laptop run

From a terminal in the repository:

```bash
python app.py --local
```

Open:

```text
http://127.0.0.1:7860
```

Local laptop mode does not need Gradio Live or a public share URL.

## How to run in Colab

1. Open this notebook in Google Colab.
2. Select **Runtime > Change runtime type > GPU** when you want model loading or LoRA/QLoRA training.
3. Run cells from top to bottom.
4. Validate a dataset before training.
5. Launch the Gradio app cell with `share=True` because Colab localhost is not directly reachable from your browser.
6. Download adapter ZIP after a successful LoRA/QLoRA run.

## Demo flow for lecturer

1. Show dependency installation and GPU check.
2. Load helper modules and default model configuration.
3. Run sample Mind Map and Venn validation.
4. Launch Gradio.
5. Upload curated dataset.
6. Start a small LoRA smoke training only if GPU is available.
7. Refresh training status until adapter output appears.
8. Generate a before/after Mermaid sample and show validation plus rendered preview.


# 2. Install Dependencies

This cell is Colab-friendly. If the notebook is opened without the repository files, it clones the GitHub repo first, then installs `requirements.txt`. In local usage, it uses the current project folder.

# 3. Restart Runtime Note

If pip upgrades packages that were already imported in the current Python runtime, restart the Colab runtime and run the notebook again from the first cell. This prevents mixed old/new module versions.

# 4. Clone/Load Project Files if Needed

The next code cell checks whether `src/mermaid_generate/` exists. If not, it clones the GitHub repository and changes into the project folder. This keeps the notebook runnable both from Colab and from an already-cloned local checkout.


In [3]:
from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/Justindwinata/MermaidGenerate.git"
PROJECT_DIR = Path.cwd()

if not (PROJECT_DIR / "src" / "mermaid_generate").exists():
    if PROJECT_DIR.name != "MermaidGenerate":
        !git clone {REPO_URL} MermaidGenerate
        os.chdir("MermaidGenerate")
        PROJECT_DIR = Path.cwd()

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

%pip install -q -r requirements.txt
print("Project directory:", PROJECT_DIR)
print("Source directory:", SRC_DIR)


Project directory: /content/MermaidGenerate
Source directory: /content/MermaidGenerate/src


# 5. Import Libraries

Core imports for dataset validation, Mermaid preview, Transformers/PyTorch inference, fine-tuning, adapter activation, evaluation, and Gradio.


In [4]:
from pathlib import Path
import json
import time

from mermaid_generate.config import DEFAULT_MODEL_ID, MERMAID_JS_VERSION
from mermaid_generate.dataset_validator import validate_dataset_file
from mermaid_generate.inference import GenerationSettings, generate_mermaid
from mermaid_generate.mermaid_preview import build_mermaid_preview_html
from mermaid_generate.mermaid_validator import validate_mermaid_code, postprocess_mermaid_output
from mermaid_generate.model_loader import ACTIVE_MODEL_STATE, load_base_model, load_adapter, load_full_model
from mermaid_generate.training import FineTuningConfig, run_fine_tuning
from mermaid_generate.training_manager import TRAINING_MANAGER
from mermaid_generate.adapter_manager import current_adapter_metadata, activate_training_result
from mermaid_generate.evaluation import evaluate_predictions, run_manual_test_set

print("Mermaid.js pinned version:", MERMAID_JS_VERSION)
print("Default model:", DEFAULT_MODEL_ID)


Mermaid.js pinned version: 11.13.0
Default model: TinyLlama/TinyLlama-1.1B-Chat-v1.0


# 6. GPU Check

Use this cell to confirm whether CUDA is available. GPU is recommended for model loading and required for practical LoRA/QLoRA training. CPU can still run dataset validation and some lightweight checks.


In [5]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA runtime:", torch.version.cuda)
    print("BF16 supported:", torch.cuda.is_bf16_supported())
else:
    print("No CUDA GPU detected. Use CPU only for lightweight validation/demo, or switch Colab runtime to GPU for training.")


CUDA available: True
GPU: Tesla T4
CUDA runtime: 12.8
BF16 supported: True


# 4. Configuration

Adjust these values before loading the model or starting fine-tuning. TinyLlama is used because it is local-friendly, LLaMA-family, Transformers/PyTorch-compatible, and suitable for LoRA/QLoRA experiments.


In [6]:
MODEL_ID = DEFAULT_MODEL_ID
USE_4BIT_FOR_BASE_INFERENCE = False
LOAD_ADAPTER_PATH = None

DEFAULT_GENERATION = GenerationSettings(
    max_new_tokens=320,
    temperature=0.2,
    top_p=0.9,
    repetition_penalty=1.05,
)

print({
    "MODEL_ID": MODEL_ID,
    "USE_4BIT_FOR_BASE_INFERENCE": USE_4BIT_FOR_BASE_INFERENCE,
    "LOAD_ADAPTER_PATH": LOAD_ADAPTER_PATH,
})


{'MODEL_ID': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0', 'USE_4BIT_FOR_BASE_INFERENCE': False, 'LOAD_ADAPTER_PATH': None}


# 5. Reference Notebook Notes

The project inspected both reference notebooks and documented the decision in `docs/REFERENCE_NOTEBOOK_INSPECTION.md`. The first notebook is useful for Mermaid rendering and browser job patterns but uses llama.cpp/GGUF and flowcharts. The second notebook is useful for Transformers/PyTorch fine-tuning UI patterns but uses Qwen identity and flowchart-oriented logic. MermaidGenerate keeps the useful engineering patterns while targeting Mind Map and Venn only.


In [7]:
inspection_path = Path("docs/REFERENCE_NOTEBOOK_INSPECTION.md")
print(inspection_path.read_text(encoding="utf-8")[:3000])


# Reference Notebook Inspection

Inspection date: 2026-07-22

## `2fa1391d-0227-4bdc-b8d0-13379ae145a6.ipynb`

This notebook contains a flowchart-focused Mermaid generator. It installs `llama-cpp-python`, detects the Colab CUDA environment, loads a GGUF Gemma wrapper model, then exposes a Flask-based web app with Mermaid v11 rendering and Cloudflare Tunnel access.

Useful implementation references:

- Colab environment readiness checks.
- Background generation job structure.
- Mermaid output cleanup before rendering.
- Browser rendering pattern using a pinned Mermaid script.
- User-facing render error handling.

Items intentionally not copied into MermaidGenerate:

- The primary llama.cpp/GGUF runtime.
- Gemma4 GGUF model identity.
- Flowchart target behavior.
- Flask-first app structure.

Reason: MG-0001 requires Transformers and PyTorch inference as the primary runtime and restricts final generation targets to Mind Map and Venn Diagram.

## `llm-012-qwen2.5-1.5b-web-upload-finetuning

# 7. Dataset Upload and Validation

This section validates JSON/JSONL datasets before training. Accepted formats are messages, prompt-completion, and instruction-output with optional input.


In [8]:
try:
    from google.colab import files
    uploaded = files.upload()
    DATASET_PATH = Path(next(iter(uploaded))) if uploaded else None
except Exception:
    DATASET_PATH = Path("datasets/examples/mindmap_examples.jsonl") if Path("datasets/examples/mindmap_examples.jsonl").exists() else None

if DATASET_PATH:
    report = validate_dataset_file(DATASET_PATH)
    print(report.as_dict())
    VALID_DATA = report.valid_data
else:
    print("No dataset uploaded yet.")
    VALID_DATA = []


Saving mixed_mindmap_venn_curated.jsonl to mixed_mindmap_venn_curated.jsonl
{'total_samples': 150, 'valid_samples': 150, 'invalid_samples': 0, 'warning_samples': 0, 'diagram_type_counts': {'mindmap': 75, 'venn': 75}, 'source_format_counts': {'prompt_completion': 150}, 'duplicate_count': 0, 'invalid_rows': [], 'warnings': [], 'normalized_preview': [{'id': '2594a49fb8c373f4', 'diagram_type': 'mindmap', 'prompt': 'Buat mind map tentang strategi belajar matematika.', 'target': 'mindmap\n  root((Strategi Belajar Matematika))\n    Konsep\n      Definisi\n      Rumus\n    Latihan\n      Soal Harian\n      Diskusi\n    Evaluasi\n      Kuis\n      Refleksi', 'source_format': 'prompt_completion', 'language': 'id', 'domain': 'education', 'complexity': 'simple'}, {'id': 'b84e0350c755ee96', 'diagram_type': 'mindmap', 'prompt': 'Buat mind map terstruktur tentang strategi belajar matematika untuk presentasi tugas.', 'target': 'mindmap\n  root((Strategi Belajar Matematika 2))\n    Konsep\n      Defini

# 8. Mermaid Validation and Preview

This section checks Mind Map and Venn syntax and verifies that the preview renderer converts assignment-facing `venn` code into Mermaid's renderer-facing `venn-beta` syntax.


In [9]:
mindmap_example = """mindmap
  root((Online Learning))
    Platforms
      LMS
      Video Class
    Activities
      Quiz
      Discussion"""
venn_example = """venn
  set A["Students"]
  set B["Workers"]
  union A,B
    text "Working Students"""

for sample in [mindmap_example, venn_example]:
    result = validate_mermaid_code(sample)
    print(result.message())
    display({"normalized_code": result.normalized_code, "renderer_code": result.renderer_code})


Valid mindmap Mermaid code.


{'normalized_code': 'mindmap\n  root((Online Learning))\n    Platforms\n      LMS\n      Video Class\n    Activities\n      Quiz\n      Discussion',
 'renderer_code': 'mindmap\n  root((Online Learning))\n    Platforms\n      LMS\n      Video Class\n    Activities\n      Quiz\n      Discussion'}

Valid venn Mermaid code.


{'normalized_code': 'venn\n  set A["Students"]\n  set B["Workers"]\n  union A,B\n    text "Working Students',
 'renderer_code': 'venn-beta\n  set A["Students"]\n  set B["Workers"]\n  union A,B\n    text "Working Students'}

# 9. Model Loading

This section loads the Transformers tokenizer/model and keeps the active model state used by inference. The runtime remains Transformers + PyTorch, not llama.cpp.


In [10]:
try:
    if LOAD_ADAPTER_PATH:
        state = load_adapter(LOAD_ADAPTER_PATH, model_id=MODEL_ID, use_4bit=USE_4BIT_FOR_BASE_INFERENCE)
    else:
        state = load_base_model(MODEL_ID, use_4bit=USE_4BIT_FOR_BASE_INFERENCE)
    print(state.status_text())
except Exception as exc:
    print("Model loading failed:", repr(exc))
    print("Recommendation: use GPU runtime, LoRA/QLoRA, smaller sequence length, or smaller batch size.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Active base model: TinyLlama/TinyLlama-1.1B-Chat-v1.0


# 10. Inference

This section runs Mermaid inference and post-processing. Raw model output is extracted, repaired if needed, validated, and then rendered through the Mermaid iframe preview.


In [11]:
def generate_demo(prompt, diagram_type="Auto Detect"):
    return generate_mermaid(
        prompt,
        diagram_type,
        DEFAULT_GENERATION,
    )

# Example after model loading:
# result = generate_demo("Create a mind map about digital marketing for UMKM", "Mind Map")
# print(result.code)
# display(build_mermaid_preview_html(result.code))


# 11. Fine-Tuning Modes

This section defines the smoke fine-tuning configuration. The app supports LoRA, QLoRA 4-bit, and Full Fine-Tuning from the browser UI.


In [12]:
TRAINING_CONFIG = FineTuningConfig(
    mode="LoRA",
    model_id=MODEL_ID,
    epochs=1,
    learning_rate=2e-4,
    batch_size=1,
    gradient_accumulation=4,
    max_seq_length=1024,
    validation_split=0.1,
)
TRAINING_CONFIG.validate()
print(TRAINING_CONFIG)


FineTuningConfig(mode='LoRA', model_id='TinyLlama/TinyLlama-1.1B-Chat-v1.0', output_root='outputs', epochs=1, learning_rate=0.0002, batch_size=1, gradient_accumulation=4, max_seq_length=1024, validation_split=0.1, lora_r=16, lora_alpha=32, lora_dropout=0.05, seed=42)


# 11. LoRA / QLoRA / Full Fine-Tuning

Use this section only after `VALID_DATA` contains validated samples. Training is real. For QLoRA, CUDA and bitsandbytes compatibility are required.

## Recommended smoke demo

For assignment demo, use `configs/lora_smoke_config.json` with `datasets/curated/mixed_mindmap_venn_curated.jsonl`:

- LoRA
- max samples: 32
- epochs: 1
- batch size: 1
- gradient accumulation: 4
- max sequence length: 512
- learning rate: 2e-4
- validation split: 0.1

In the Gradio UI, upload the curated mixed dataset, validate it, enter these values, then start training from the browser. Do not run or claim success if no GPU is available.


In [13]:
import json
from pathlib import Path

SMOKE_CONFIG_PATH = Path("configs/lora_smoke_config.json")
if SMOKE_CONFIG_PATH.exists():
    smoke_config = json.loads(SMOKE_CONFIG_PATH.read_text(encoding="utf-8"))
    print("LoRA smoke config:", smoke_config)
else:
    smoke_config = None

def run_training_once(valid_data, config):
    logs = []
    def progress(payload):
        logs.append(payload)
        print(payload)
    result = run_fine_tuning(valid_data, config, report_progress=progress)
    print(result)
    return result, logs

# Example after validating curated data and confirming GPU availability:
# small_data = VALID_DATA[: smoke_config["max_samples"]]
# smoke_training_config = FineTuningConfig(
#     mode="LoRA",
#     model_id=MODEL_ID,
#     epochs=smoke_config["epochs"],
#     learning_rate=smoke_config["learning_rate"],
#     batch_size=smoke_config["batch_size"],
#     gradient_accumulation=smoke_config["gradient_accumulation"],
#     max_seq_length=smoke_config["max_sequence_length"],
#     validation_split=smoke_config["validation_split"],
# )
# training_result, training_logs = run_training_once(small_data, smoke_training_config)


LoRA smoke config: {'mode': 'LoRA', 'dataset_path': 'datasets/curated/mixed_mindmap_venn_curated.jsonl', 'max_samples': 32, 'epochs': 1, 'batch_size': 1, 'gradient_accumulation': 4, 'max_sequence_length': 512, 'learning_rate': 0.0002, 'validation_split': 0.1, 'model_id': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0', 'expected_output_root': 'outputs/adapters/', 'notes': ['Use a Colab GPU runtime for this smoke test.', 'Do not claim training success unless a real run completes.', 'If CUDA OOM occurs, reduce max_samples or max_sequence_length.']}


# 12. Training Manager, Logs, Cancel, and Adapter Activation

The Gradio UI uses a background training manager. It exposes honest states: idle, validating_dataset, loading_model, training, cancelling, completed, and failed. Cancellation is checked through a callback at safe training boundaries.


In [14]:
def start_managed_training(valid_data, config):
    job_id = TRAINING_MANAGER.start(valid_data, config)
    print("Started job:", job_id)
    return job_id

def show_job(job_id):
    state = TRAINING_MANAGER.get(job_id)
    print(state.as_dict() if state else "Job not found")
    return state

# Example:
# job_id = start_managed_training(VALID_DATA, TRAINING_CONFIG)
# show_job(job_id)
# TRAINING_MANAGER.cancel(job_id)


# 13. Adapter ZIP Download

LoRA and QLoRA training outputs are saved under `outputs/adapters/<timestamp>/` and zipped for download. Full fine-tuned models are saved under `outputs/full_models/<timestamp>/`. Successful adapter/model outputs can be activated without restarting the app where resources allow.


In [15]:
# Example after a completed TrainingResult:
# metadata = activate_training_result(training_result, model_id=MODEL_ID)
# print(metadata.as_dict())
# from google.colab import files
# if training_result.zip_path:
#     files.download(training_result.zip_path)
print(current_adapter_metadata().as_dict())


{'active': True, 'mode': 'base', 'model_id': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0', 'adapter_path': None, 'full_model_path': None, 'zip_path': None, 'status': 'Active base model: TinyLlama/TinyLlama-1.1B-Chat-v1.0'}


# 12. Gradio Web App

This builds the browser UI with two tabs: **Generator Mermaid** and **Dataset & Fine-Tuning**.

# 13. Local vs Colab Access Explanation

- Local laptop: run `python app.py --local` from the terminal and open `http://127.0.0.1:7860`.
- Google Colab: use the next notebook launch cell with `share=True` and open the generated `https://xxxxx.gradio.live` URL.
- Do not open `0.0.0.0:7860` in your browser. `0.0.0.0` is a server bind address, not a browser URL.
- `127.0.0.1` only works when the app is running on the same machine as your browser.

## Demo checklist inside the UI

1. Open **Generator Mermaid** and run one Mind Map prompt.
2. Run one Venn prompt and note that preview uses Mermaid `venn-beta` internally.
3. Confirm the rendered SVG diagram appears in the iframe preview panel.
4. Open **Dataset & Fine-Tuning**.
5. Upload `datasets/curated/mixed_mindmap_venn_curated.jsonl`.
6. Confirm validation reports 150 valid samples and 0 invalid samples.
7. Enter the LoRA smoke settings from `configs/lora_smoke_config.json` only if GPU is available.


In [16]:
from app import LaunchConfig, build_app, print_launch_banner

demo = build_app()
print("Gradio app object ready. Run the next cell to launch in Colab/share mode.")
print("For a local laptop terminal, use: python app.py --local")


/content/MermaidGenerate/app.py:284: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(title=APP_TITLE, css=APP_CSS) as demo:


Gradio app object ready. Run the next cell to launch.


# 14. Demo Steps

Run the next cell to launch in Google Colab. Keep the cell running while using the web app.

Expected Colab output:

- a temporary public Gradio URL ending in `gradio.live`
- app opens with **Generator Mermaid** and **Dataset & Fine-Tuning** tabs
- Mermaid preview renders inside the iframe panel
- dataset validation is visible before any training attempt
- training logs/progress update through the Refresh Status button

Important: in Colab, open the `gradio.live` URL. Do not open `0.0.0.0:7860` in your browser.

For local laptop usage, do not use this notebook launch cell. Use:

```bash
python app.py --local
```

Then open `http://127.0.0.1:7860`.

If launch fails because a port is busy, change `server_port` or run `python app.py --local --port 7861`.


In [17]:
colab_config = LaunchConfig(mode="colab", host="0.0.0.0", port=7860, share=True)
print_launch_banner(colab_config)
demo.launch(server_name=colab_config.host, server_port=colab_config.port, share=colab_config.share)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1e5cc3e53e3297db95.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# 16. Evaluation Utilities

Evaluation does not rely only on training loss. It computes syntax validity rate, diagram type accuracy, prefix accuracy, exact match after normalization, invalid output count, and markdown fence violation count.


In [18]:
manual_refs = [
    {
        "diagram_type": "mindmap",
        "prompt": "Create a mind map about online learning.",
        "target": mindmap_example,
    },
    {
        "diagram_type": "venn",
        "prompt": "Create a Venn diagram comparing students and workers.",
        "target": venn_example,
    },
]
manual_predictions = [mindmap_example, venn_example]
metrics = evaluate_predictions(manual_refs, manual_predictions)
print(metrics.as_dict())


{'total': 2, 'syntax_validity_rate': 1.0, 'diagram_type_accuracy': 1.0, 'prefix_accuracy': 1.0, 'exact_match_rate': 1.0, 'invalid_output_count': 0, 'markdown_fence_violation_count': 0}


# 15. Troubleshooting

## Evidence capture note

After launching Gradio in Colab, capture real screenshots only from the live notebook/UI session. Do not create placeholder screenshots. Save screenshot evidence under `docs/evidence/screenshots/` and update `docs/evidence/SCREENSHOT_EVIDENCE_REPORT.md`.

## Expected outputs

- Local laptop app is available at `http://127.0.0.1:7860` when running `python app.py --local`.
- Colab app provides a temporary public Gradio URL when launched with `share=True`.
- Dataset validation summary with valid/invalid counts.
- Mermaid code that starts with `mindmap` or `venn`.
- Rendered Mermaid SVG preview in the browser UI iframe panel.
- Training status, progress, loss logs, and result summary during fine-tuning.
- Adapter directory and ZIP path after successful LoRA/QLoRA training.

## Common errors

- **Do not open `0.0.0.0:7860`:** `0.0.0.0` is only a server bind address. In Colab, open the `gradio.live` link. Locally, open `http://127.0.0.1:7860`.
- **No CUDA GPU detected:** switch Colab runtime to GPU for training.
- **CUDA out of memory:** reduce batch size, max sequence length, or use LoRA/QLoRA instead of Full Fine-Tuning.
- **bitsandbytes import/load error:** QLoRA needs a compatible CUDA/bitsandbytes runtime; use LoRA if compatibility fails.
- **Gradio port busy:** change the port or restart the runtime.
- **Local URL not reachable in Colab:** use the public Gradio share link; Colab localhost is not directly exposed to your laptop browser.
- **Mermaid preview shows text instead of a diagram:** restart the app after pulling the latest code; the fixed preview uses an iframe renderer.
- **Model output is invalid:** check the final validated Mermaid code; the fallback repair layer may produce a safe diagram even when raw output is noisy.
- **Dataset path error in scripts:** use paths relative to repo root or absolute paths; `scripts/validate_curated_datasets.py` handles both.
- **Adapter ZIP does not appear:** refresh training status after completed LoRA/QLoRA training and verify the job finished successfully.
- **Venn render error:** Venn depends on Mermaid.js v11 support; the app renders assignment-facing `venn` code internally as Mermaid `venn-beta`.

# 16. Limitations

- Inference uses Transformers and PyTorch as required.
- llama.cpp/GGUF is optional future compatibility only unless implemented later.
- Local mode requires dependencies installed on the local machine.
- Local model loading may require enough RAM or VRAM.
- Model quality depends on dataset size and training quality.
- Runtime repair may be used to guarantee valid Mermaid syntax.
- The example datasets are small and are not enough for strong model quality.
- Full fine-tuning may fail on limited GPU memory.
- QLoRA requires a compatible CUDA/bitsandbytes environment.
- Venn preview depends on Mermaid.js Venn support; this project pins Mermaid v11 and renders Venn via `venn-beta`.
- No paid API is used.
- Training progress and success are real, not simulated.
- Colab runtime resets can clear loaded models; save adapters/models before ending the session.

# 17. Submission Guide

Submit the notebook, source code, curated dataset, documentation, and GitHub repository link. Adapter ZIP files may be generated during the demo and should not be committed if large.
